In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

def safe_proba_or_score(estimator, X, cv, n_jobs=-1):
    """
    Try to get cross-validated probability scores for the positive class.
    Falls back to decision_function if predict_proba not available.
    Returns array of shape (n_samples,) with score-like values for ROC AUC.
    """
    if hasattr(estimator, "predict_proba"):
        proba = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba", n_jobs=n_jobs)
        return proba[:, 1]
    elif hasattr(estimator, "decision_function"):
        scores = cross_val_predict(estimator, X, y, cv=cv, method="decision_function", n_jobs=n_jobs)
        return scores
    else:
        # last resort: use predicted labels (not ideal for ROC AUC)
        preds = cross_val_predict(estimator, X, y, cv=cv, n_jobs=n_jobs)
        return preds  # 0/1 labels

def evaluate_train_test(estimator, X, y, test_size=0.2, random_state=42):
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)
    clf = estimator
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_val)

    # ROC AUC: try predict_proba or decision_function; if none, compute on labels (will be poor)
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_val)[:, 1]
    elif hasattr(clf, "decision_function"):
        y_score = clf.decision_function(X_val)
    else:
        y_score = y_pred

    return {
        "accuracy": accuracy_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred),
        "roc_auc": roc_auc_score(y_val, y_score)
    }

def evaluate_kfold(estimator, X, y, n_splits=5, random_state=42, n_jobs=-1):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    # get cross-validated label predictions for accuracy & f1
    y_pred_cv = cross_val_predict(estimator, X, y, cv=cv, n_jobs=n_jobs)

    # get cross-validated scores for ROC AUC (proba or decision_function if available)
    if hasattr(estimator, "predict_proba"):
        y_score_cv = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba", n_jobs=n_jobs)[:, 1]
    elif hasattr(estimator, "decision_function"):
        y_score_cv = cross_val_predict(estimator, X, y, cv=cv, method="decision_function", n_jobs=n_jobs)
    else:
        # fallback: use label predictions (not recommended)
        y_score_cv = y_pred_cv

    return {
        "accuracy": accuracy_score(y, y_pred_cv),
        "f1": f1_score(y, y_pred_cv),
        "roc_auc": roc_auc_score(y, y_score_cv)
    }

if __name__ == "__main__":
    # Load preprocessed data (same filenames used in your notebooks)
    train = pd.read_csv("titanic_train.csv")
    # keep test load optional
    # test = pd.read_csv("titanic_test.csv")

    # prepare features and target (adjust if your columns differ)
    X = train.drop(["PassengerId", "Survived"], axis=1)
    y = train["Survived"]

    models = {
        "SVM": SVC(random_state=42, probability=True),
        "RandomForest": RandomForestClassifier(random_state=42),
        "KNN": KNeighborsClassifier()
    }

    results = []
    for name, model in models.items():
        print(f"Evaluating {name} ...")

        tt_metrics = evaluate_train_test(model, X, y)
        kf_metrics = evaluate_kfold(model, X, y, n_splits=5)

        results.append({
            "model": name,
            "eval": "train_test_split",
            "accuracy": tt_metrics["accuracy"],
            "f1": tt_metrics["f1"],
            "roc_auc": tt_metrics["roc_auc"]
        })
        results.append({
            "model": name,
            "eval": "kfold_cv",
            "accuracy": kf_metrics["accuracy"],
            "f1": kf_metrics["f1"],
            "roc_auc": kf_metrics["roc_auc"]
        })

    df_results = pd.DataFrame(results)
    print("\nSummary:")
    print(df_results.pivot(index="model", columns="eval", values=["accuracy", "f1", "roc_auc"]))
    # also save to CSV
    df_results.to_csv("model_evaluation_summary.csv", index=False)
    print("\nSaved summary to model_evaluation_summary.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'titanic_train.csv'